# Example 11.4 / 3: Distinguish Failure Modes

Companion notebook for **11.4 Pitfalls of Nonlinear Programming** in the AMPL lab Documentation, section "Distinguish Failure Modes". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Distinguish Failure Modes

| Symptom | Possible cause | Evidence to inspect | Concrete instance |
|---|---|---|---|
| evaluation error before iterations | invalid start or expression outside its domain | starts, propagated bounds, function arguments | `var x := 0; minimize F: x - log(x);` with Ipopt gives status `failure`; the example below fixes the start |
| infeasible result | inconsistent constraints or failure to find feasibility | original-unit residuals, alternate starts, independent feasibility model | `x^2 + y^2 <= 1` with `x + y >= 3`: the line's distance from the origin, $3/\sqrt2\approx2.12$, exceeds the radius 1, so no point satisfies both, whatever a local solver reports |
| slow or erratic progress | scaling, degeneracy, poor derivatives, flat or steep regions | iteration log, derivative check, scaled magnitudes | balances in dollars near $10^9$: residual resolution about $4.77\times10^{-7}$, see "Scaling" |
| answers change with the start | multiple stationary points or algorithmic failure | objectives and residuals for every run; global bound if required | the `Wells` runs under "Starting Points and Local Optima" |
| `solved` but unusable decision | model or data error | units, omitted constraints, scenario and business-rule checks | the congestion model without `ServeDemand`: `solved` with `Flow = 0` |
| clipping changes the model | an argument clipped to avoid an error | whether the clipped relation is intended | `log(max(x, 0.001))` at $x=0$ returns `-6.90776` instead of an error, silently changing the function |

The evaluation error and its repair. A start at `0` puts `log(x)` outside its domain; a start at
`1`, inside the domain, lets Ipopt reach the minimizer $x=1$ of $x-\log x$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x := 0;  # start 0 is outside the domain of log

minimize F: x - log(x);  # defined only for x > 0

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print("start 0:", ampl.solve_result)
ampl.var["x"].set_value(1)
ampl.solve(solver="ipopt", verbose=False)
print("start 1:", ampl.solve_result, ampl.var["x"].value(), ampl.obj["F"].value())

start 0: failure
start 1: solved 1.0 1.0
